# Quantization

## Learning Objectives
1. Implement INT8 quantization from scratch: scale, zero-point, quantize/dequantize
2. Apply PyTorch dynamic and static quantization; compare size and speed
3. Apply post-training quantization (PTQ) to a BERT-style model
4. Compare INT8 vs INT4 accuracy-compression trade-offs


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import copy, time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — INT8 Quantization from Scratch

In [ ]:
# ── Level 1: manual INT8 quantize / dequantize ───────────────────────────
import numpy as np

np.random.seed(0)
W = np.random.randn(4, 8).astype(np.float32)  # simulated weight matrix
print("Original W (row 0):", W[0])

def quantize_int8(x: np.ndarray):
    """Symmetric per-tensor INT8 quantization."""
    scale     = np.max(np.abs(x)) / 127.0     # maps [-max, +max] → [-127, 127]
    zero_point = 0                              # symmetric: ZP = 0
    x_q = np.clip(np.round(x / scale), -128, 127).astype(np.int8)
    return x_q, scale, zero_point

def dequantize_int8(x_q: np.ndarray, scale: float, zero_point: int = 0):
    """Recover approximate float from INT8."""
    return (x_q.astype(np.float32) - zero_point) * scale

W_q, scale, zp = quantize_int8(W)
W_reconstructed = dequantize_int8(W_q, scale, zp)

# Measure quantization error
max_err  = np.max(np.abs(W - W_reconstructed))
mean_err = np.mean(np.abs(W - W_reconstructed))
print(f"\nScale={scale:.5f}  ZeroPoint={zp}")
print(f"Quantized W_q (row 0): {W_q[0]}")
print(f"Reconstructed  (row 0): {W_reconstructed[0]}")
print(f"Max error: {max_err:.6f}  Mean error: {mean_err:.6f}")

# Asymmetric (affine) quantization
def quantize_uint8_affine(x: np.ndarray):
    """Affine (asymmetric) UINT8: covers [x_min, x_max] → [0, 255]."""
    x_min, x_max = x.min(), x.max()
    scale     = (x_max - x_min) / 255.0
    zero_point = int(np.round(-x_min / scale))
    x_q = np.clip(np.round(x / scale + zero_point), 0, 255).astype(np.uint8)
    return x_q, scale, zero_point

W_q2, s2, zp2 = quantize_uint8_affine(W)
W_rec2 = (W_q2.astype(np.float32) - zp2) * s2
print(f"\nAffine UINT8: scale={s2:.5f}  zp={zp2}")
print(f"Mean error affine: {np.mean(np.abs(W - W_rec2)):.6f}")


## Level 2 — PyTorch Dynamic and Static Quantization

In [ ]:
# ── Level 2: torch dynamic + static quantization ─────────────────────────
import torch, time, copy
import torch.nn as nn

class SimpleLM(nn.Module):
    """Small LSTM-based model suitable for dynamic quantization."""
    def __init__(self, vocab=100, embed=32, hidden=64, n_class=10):
        super().__init__()
        self.embed = nn.Embedding(vocab, embed)
        self.lstm  = nn.LSTM(embed, hidden, batch_first=True)
        self.fc    = nn.Linear(hidden, n_class)

    def forward(self, x):
        e = self.embed(x)
        _, (h, _) = self.lstm(e)
        return self.fc(h.squeeze(0))

model = SimpleLM().cpu()   # quantization APIs require CPU

def model_size_mb(m):
    buf = 0
    for p in m.parameters():
        buf += p.nelement() * p.element_size()
    return buf / (1024 ** 2)

def eval_speed(m, x, repeats=200):
    m.eval()
    with torch.no_grad():
        for _ in range(10):           # warmup
            m(x)
        t0 = time.perf_counter()
        for _ in range(repeats):
            m(x)
    return (time.perf_counter() - t0) / repeats * 1000   # ms

x_in = torch.randint(0, 100, (32, 20))   # (batch=32, seq=20)

# Baseline
size_fp32 = model_size_mb(model)
speed_fp32 = eval_speed(model, x_in)
print(f"FP32   size={size_fp32:.4f} MB  speed={speed_fp32:.3f} ms")

# Dynamic quantization (LSTM + Linear)
try:
    model_dyn = torch.quantization.quantize_dynamic(
        copy.deepcopy(model),
        {nn.LSTM, nn.Linear},
        dtype=torch.qint8
    )
    speed_dyn = eval_speed(model_dyn, x_in)
    size_dyn  = model_size_mb(model_dyn)
    print(f"INT8 dynamic size={size_dyn:.4f} MB  speed={speed_dyn:.3f} ms  "
          f"speedup={speed_fp32/speed_dyn:.2f}x")
except Exception as e:
    print(f"Dynamic quantization: {e}")

# Static post-training quantization (MLP — LSTMs need special handling)
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant   = torch.quantization.QuantStub()
        self.fc1     = nn.Linear(64, 128)
        self.fc2     = nn.Linear(128, 10)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return self.dequant(x)

import torch.nn.functional as F
mlp = MLP()
mlp.eval()
mlp.qconfig = torch.quantization.get_default_qconfig("fbgemm")
torch.quantization.prepare(mlp, inplace=True)
calib_data = torch.randn(128, 64)
with torch.no_grad():
    mlp(calib_data)
torch.quantization.convert(mlp, inplace=True)
x64 = torch.randn(32, 64)
print(f"Static INT8 MLP output shape: {mlp(x64).shape}")
print("Static quantization complete — model ready for INT8 inference")


## Real-World Example 1 — Post-Training Quantization on BERT-style Model

In [ ]:
# ── RW1: PTQ on a BERT-style encoder ─────────────────────────────────────
import torch, copy, time
import torch.nn as nn

# Simulate a BERT encoder block (no download needed)
class BERTBlock(nn.Module):
    def __init__(self, hidden=256, n_heads=4, ff_dim=512):
        super().__init__()
        self.attn   = nn.MultiheadAttention(hidden, n_heads, batch_first=True)
        self.norm1  = nn.LayerNorm(hidden)
        self.ff     = nn.Sequential(nn.Linear(hidden, ff_dim), nn.GELU(),
                                    nn.Linear(ff_dim, hidden))
        self.norm2  = nn.LayerNorm(hidden)
        self.cls_fc = nn.Linear(hidden, 2)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return self.cls_fc(x[:, 0])    # CLS token → classification logits

model_fp32 = BERTBlock().cpu()
model_q    = copy.deepcopy(model_fp32)

# Apply dynamic quantization (Linear layers)
model_q = torch.quantization.quantize_dynamic(
    model_q, {nn.Linear}, dtype=torch.qint8
)

# Size comparison
def param_bytes(m):
    return sum(p.nelement() * p.element_size() for p in m.parameters())

sz_fp32 = param_bytes(model_fp32) / 1024
sz_q    = param_bytes(model_q)    / 1024
print(f"FP32 size: {sz_fp32:.1f} KB")
print(f"INT8 size: {sz_q:.1f} KB")
print(f"Compression: {sz_fp32/sz_q:.2f}×")

# Simulate accuracy proxy: cosine similarity between FP32 and INT8 outputs
x_test = torch.randn(16, 32, 256)   # (batch, seq_len, hidden)
with torch.no_grad():
    out_fp = model_fp32(x_test)
    out_q  = model_q(x_test)
cos_sim = torch.nn.functional.cosine_similarity(out_fp, out_q, dim=-1).mean().item()
print(f"\nOutput cosine similarity (FP32 vs INT8): {cos_sim:.4f}")
print("(Values close to 1.0 indicate PTQ introduces minimal accuracy drop)")

# Inference speed
def latency_ms(m, x, n=100):
    with torch.no_grad():
        for _ in range(5): m(x)
        t0 = time.perf_counter()
        for _ in range(n): m(x)
    return (time.perf_counter() - t0) / n * 1000

lat_fp = latency_ms(model_fp32, x_test)
lat_q  = latency_ms(model_q,    x_test)
print(f"FP32 latency: {lat_fp:.2f} ms  |  INT8 latency: {lat_q:.2f} ms  "
      f"|  speedup: {lat_fp/lat_q:.2f}×")


## Real-World Example 2 — Quantization-Aware Training (QAT)

In [ ]:
# ── RW2: QAT — recover accuracy lost in PTQ ──────────────────────────────
import torch, copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

# Synthetic task
X = torch.randn(1000, 32)
y = (X[:, 0] - X[:, 1] > 0).long()
train_ld = DataLoader(TensorDataset(X[:800], y[:800]), batch_size=64, shuffle=True)
X_val, y_val = X[800:], y[800:]

class QMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant   = torch.quantization.QuantStub()
        self.fc1     = nn.Linear(32, 128)
        self.fc2     = nn.Linear(128, 64)
        self.fc3     = nn.Linear(64, 2)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return self.dequant(x)

def train_eval(model, epochs=15, qat=False):
    model = copy.deepcopy(model)
    if qat:
        model.train()
        model.qconfig = torch.quantization.get_default_qat_qconfig("fbgemm")
        torch.quantization.prepare_qat(model, inplace=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for xb, yb in train_ld:
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
    if qat:
        model.eval()
        torch.quantization.convert(model, inplace=True)
    model.eval()
    with torch.no_grad():
        acc = (model(X_val).argmax(1) == y_val).float().mean().item()
    return model, acc

# FP32 baseline
_, acc_fp32 = train_eval(QMLP())
print(f"FP32 training accuracy: {acc_fp32:.3f}")

# PTQ: train FP32, then quantize (no recovery)
fp32_model, _ = train_eval(QMLP())
fp32_model.eval()
fp32_model.qconfig = torch.quantization.get_default_qconfig("fbgemm")
torch.quantization.prepare(copy.deepcopy(fp32_model), inplace=True)
ptq_model = copy.deepcopy(fp32_model)
torch.quantization.prepare(ptq_model, inplace=True)
with torch.no_grad():
    ptq_model(X[:200])   # calibration
torch.quantization.convert(ptq_model, inplace=True)
ptq_model.eval()
with torch.no_grad():
    acc_ptq = (ptq_model(X_val).argmax(1) == y_val).float().mean().item()
print(f"PTQ accuracy:           {acc_ptq:.3f}")

# QAT: quantize during training
_, acc_qat = train_eval(QMLP(), qat=True)
print(f"QAT accuracy:           {acc_qat:.3f}")
print("\nQAT typically recovers accuracy lost by PTQ on small models.")


## Real-World Example 3 — INT8 vs INT4 Accuracy-Compression Trade-off

In [ ]:
# ── RW3: INT8 vs INT4 — accuracy drop vs compression gain ────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

# Simulate post-training weight quantization at different bit widths
def quantize_to_bits(W: torch.Tensor, bits: int) -> torch.Tensor:
    """Symmetric uniform quantization to `bits`-bit integers."""
    levels     = 2 ** (bits - 1) - 1              # e.g., 127 for int8, 7 for int4
    scale      = W.abs().max() / levels
    W_q        = (W / scale).round().clamp(-levels, levels)
    return W_q * scale                              # dequantized (approximate float)

# Build and train a small model
X = torch.randn(800, 32)
y = (X[:, :3].sum(1) > 0).long()
loader = DataLoader(TensorDataset(X[:600], y[:600]), batch_size=64, shuffle=True)
X_val, y_val = X[600:], y[600:]

model_fp32 = nn.Sequential(nn.Linear(32, 128), nn.ReLU(),
                            nn.Linear(128, 64),  nn.ReLU(),
                            nn.Linear(64, 2))
opt = torch.optim.Adam(model_fp32.parameters(), lr=1e-3)
for _ in range(20):
    model_fp32.train()
    for xb, yb in loader:
        opt.zero_grad()
        F.cross_entropy(model_fp32(xb), yb).backward()
        opt.step()

def apply_weight_quantization(model_src, bits):
    """Return a copy of model with all Linear weights quantized to `bits`."""
    import copy
    m = copy.deepcopy(model_src)
    for layer in m:
        if isinstance(layer, nn.Linear):
            layer.weight.data = quantize_to_bits(layer.weight.data, bits)
    return m

model_fp32.eval()
with torch.no_grad():
    acc_fp = (model_fp32(X_val).argmax(1) == y_val).float().mean().item()

results = [("FP32 (baseline)", acc_fp, 32)]
for bits in [16, 8, 4]:
    m_q = apply_weight_quantization(model_fp32, bits)
    m_q.eval()
    with torch.no_grad():
        acc = (m_q(X_val).argmax(1) == y_val).float().mean().item()
    results.append((f"INT{bits}", acc, bits))

print(f"{'Config':<20}  {'Val Acc':>8}  {'Bit Width':>10}  {'Compression':>12}")
print("-" * 58)
fp32_bits = 32
for name, acc, bits in results:
    compression = fp32_bits / bits
    drop = acc_fp - acc
    print(f"{name:<20}  {acc:>8.3f}  {bits:>10}  {compression:>11.1f}×  "
          f"drop={drop:+.3f}")

print("\nINT8: ~4× compression, minimal accuracy drop.")
print("INT4: ~8× compression, but larger accuracy drop — needs QAT or calibration.")
